In [3]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import plotly
import streamlit as st
import ollama

print("All libraries installed successfully! ✅")



All libraries installed successfully! ✅


In [5]:
import ollama

print("Ollama Python package installed successfully! ✅")

Ollama Python package installed successfully! ✅


In [6]:
from pathlib import Path

PROJECT_DIR = Path.home() / "OneDrive" / "Desktop" / "MLH"

print("Project folder:", PROJECT_DIR)
print("Folder exists:", PROJECT_DIR.exists())

Project folder: C:\Users\praja\OneDrive\Desktop\MLH
Folder exists: True


In [7]:
from pathlib import Path

PROJECT_DIR = Path.home() / "OneDrive" / "Desktop" / "MLH"
DATA_DIR = PROJECT_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Data folder:", DATA_DIR)
print("Ready! ✅")

Project folder: C:\Users\praja\OneDrive\Desktop\MLH
Data folder: C:\Users\praja\OneDrive\Desktop\MLH\data
Ready! ✅


In [8]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 200

df = pd.DataFrame({
    "EmployeeID": range(1001, 1001 + n),
    "Age": np.random.randint(21, 61, n),
    "Department": np.random.choice(
        ["IT", "HR", "Finance", "Sales", "Marketing"], n
    ),
    "Salary": np.random.randint(25000, 120000, n),
    "Experience": np.random.randint(0, 25, n),
    "Satisfaction": np.random.randint(1, 6, n),
    "Performance": np.random.choice(
        ["Low", "Medium", "High"], n
    )
})

# Demo ke liye kuch missing values add karo
df.loc[[5, 18, 42], "Salary"] = np.nan
df.loc[[12, 27], "Department"] = np.nan

# Demo ke liye duplicate row add karo
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

sample_path = DATA_DIR / "sample_employees.csv"
df.to_csv(sample_path, index=False)

print("Dataset created:", sample_path)
print("Shape:", df.shape)

display(df.head())

Dataset created: C:\Users\praja\OneDrive\Desktop\MLH\data\sample_employees.csv
Shape: (201, 7)


,EmployeeID,Age,Department,Salary,Experience,Satisfaction,Performance
0,1001,59,Sales,86389.0,7,3,Low
1,1002,49,Finance,34435.0,8,4,Medium
2,1003,35,Marketing,79340.0,6,4,High
3,1004,28,Finance,69078.0,0,1,High
4,1005,41,Sales,103832.0,2,3,Low


In [9]:
analyzer_code = '''
import pandas as pd
import numpy as np

def analyze_dataframe(df):
    """Calculate deterministic statistics and data-quality metrics."""

    rows, columns = df.shape

    missing = df.isnull().sum()
    missing_cells = int(missing.sum())

    duplicate_rows = int(df.duplicated().sum())

    numeric_df = df.select_dtypes(include="number")

    numeric_summary = {}
    for col in numeric_df.columns:
        series = numeric_df[col].dropna()

        if len(series) == 0:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        outliers = int(
            ((series < q1 - 1.5 * iqr) |
             (series > q3 + 1.5 * iqr)).sum()
        )

        numeric_summary[col] = {
            "mean": round(float(series.mean()), 2),
            "median": round(float(series.median()), 2),
            "min": float(series.min()),
            "max": float(series.max()),
            "outliers": outliers
        }

    missing_pct = (missing_cells / (rows * columns) * 100) if rows * columns else 0
    duplicate_pct = (duplicate_rows / rows * 100) if rows else 0

    health_score = max(
        0,
        min(100, round(100 - 2 * missing_pct - 1.5 * duplicate_pct))
    )

    return {
        "rows": rows,
        "columns": columns,
        "missing_cells": missing_cells,
        "missing_by_column": missing[missing > 0].to_dict(),
        "duplicate_rows": duplicate_rows,
        "missing_percentage": round(missing_pct, 2),
        "duplicate_percentage": round(duplicate_pct, 2),
        "health_score": health_score,
        "numeric_summary": numeric_summary,
        "column_types": df.dtypes.astype(str).to_dict()
    }
'''

(PROJECT_DIR / "analyzer.py").write_text(
    analyzer_code,
    encoding="utf-8"
)

print("analyzer.py created successfully! ✅")

analyzer.py created successfully! ✅


In [10]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from analyzer import analyze_dataframe

report = analyze_dataframe(df)

print("Rows:", report["rows"])
print("Columns:", report["columns"])
print("Missing cells:", report["missing_cells"])
print("Duplicate rows:", report["duplicate_rows"])
print("Data health score:", report["health_score"])
print("\nNumeric summary:")
display(pd.DataFrame(report["numeric_summary"]).T)

Rows: 201
Columns: 7
Missing cells: 5
Duplicate rows: 1
Data health score: 99

Numeric summary:


,mean,median,min,max,outliers
EmployeeID,1100.00,1100.0,1001.0,1200.0,0.0
Age,40.67,42.0,21.0,60.0,0.0
Salary,74647.65,72715.5,25412.0,119758.0,0.0
Experience,12.04,12.0,0.0,24.0,0.0
Satisfaction,2.94,3.0,1.0,5.0,0.0


In [11]:
app_code = r'''
import streamlit as st
import pandas as pd
import plotly.express as px
import json
from pathlib import Path
from analyzer import analyze_dataframe

st.set_page_config(
    page_title="OpenLens AI",
    page_icon="🔍",
    layout="wide"
)

st.title("🔍 OpenLens AI")
st.caption("Your Data. Your Evidence. | AI-powered data analysis")

uploaded_file = st.file_uploader(
    "Upload your CSV dataset",
    type=["csv"]
)

if uploaded_file is not None:
    try:
        data = pd.read_csv(uploaded_file)
    except Exception as exc:
        st.error(f"Could not read CSV: {exc}")
        st.stop()
else:
    sample_path = Path(__file__).parent / "data" / "sample_employees.csv"
    data = pd.read_csv(sample_path)
    st.info("Showing sample employee data. Upload your own CSV to analyze it.")

if data.empty:
    st.warning("The dataset is empty.")
    st.stop()

report = analyze_dataframe(data)

st.subheader("📊 Dataset Overview")

c1, c2, c3, c4 = st.columns(4)
c1.metric("Rows", report["rows"])
c2.metric("Columns", report["columns"])
c3.metric("Missing Cells", report["missing_cells"])
c4.metric("Data Health Score", f'{report["health_score"]}/100')

tab1, tab2, tab3, tab4 = st.tabs([
    "Dataset Preview",
    "Data Quality",
    "Visual Analytics",
    "AI Insights"
])

with tab1:
    st.subheader("First 20 rows")
    st.dataframe(data.head(20), use_container_width=True)

    st.subheader("Column Information")
    info = pd.DataFrame({
        "Column": data.columns,
        "Data Type": data.dtypes.astype(str).values,
        "Missing Values": data.isnull().sum().values,
        "Unique Values": data.nunique().values
    })
    st.dataframe(info, use_container_width=True)

with tab2:
    st.subheader("Missing Values")
    missing = data.isnull().sum()
    missing = missing[missing > 0]

    if len(missing):
        missing_df = missing.rename("Missing Count").reset_index()
        missing_df.columns = ["Column", "Missing Count"]
        st.dataframe(missing_df, use_container_width=True)
    else:
        st.success("No missing values found.")

    duplicates = int(data.duplicated().sum())
    st.metric("Duplicate Rows", duplicates)

    st.subheader("Numeric Statistics")
    if report["numeric_summary"]:
        st.dataframe(
            pd.DataFrame(report["numeric_summary"]).T,
            use_container_width=True
        )
    else:
        st.info("No numeric columns found.")

with tab3:
    numeric_columns = data.select_dtypes(include="number").columns.tolist()
    categorical_columns = data.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    if numeric_columns:
        selected_num = st.selectbox(
            "Choose a numeric column",
            numeric_columns
        )
        fig = px.histogram(
            data,
            x=selected_num,
            title=f"Distribution of {selected_num}"
        )
        st.plotly_chart(fig, use_container_width=True)

    if categorical_columns:
        selected_cat = st.selectbox(
            "Choose a categorical column",
            categorical_columns
        )
        counts = data[selected_cat].fillna("Missing").astype(str).value_counts()
        fig = px.bar(
            x=counts.index,
            y=counts.values,
            labels={"x": selected_cat, "y": "Count"},
            title=f"Distribution of {selected_cat}"
        )
        st.plotly_chart(fig, use_container_width=True)

with tab4:
    st.subheader("Evidence-based AI Insights")

    st.write(
        "This report uses calculated dataset statistics. "
        "AI-generated explanations will be connected through Gemma."
    )

    st.markdown("**Dataset findings**")
    st.write(f"- Dataset contains {report['rows']} rows and {report['columns']} columns.")
    st.write(f"- {report['missing_cells']} missing cells were found.")
    st.write(f"- {report['duplicate_rows']} duplicate rows were found.")

    if report["numeric_summary"]:
        summary = pd.DataFrame(report["numeric_summary"]).T
        st.dataframe(summary, use_container_width=True)

    st.warning(
        "These are deterministic statistical findings, not live AI-generated responses yet."
    )

report_json = json.dumps(report, indent=2, default=str)
st.download_button(
    "⬇️ Download Analysis Report (JSON)",
    data=report_json,
    file_name="openlens_analysis.json",
    mime="application/json"
)

st.caption("OpenLens AI | Open-source project")
'''

(PROJECT_DIR / "app.py").write_text(app_code, encoding="utf-8")

print("app.py created successfully! ✅")

app.py created successfully! ✅


In [12]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        str(PROJECT_DIR / "app.py")
    ],
    cwd=str(PROJECT_DIR)
)

print("Streamlit process started. PID:", process.pid)

Streamlit process started. PID: 11752


In [13]:
llm_client_code = '''
import ollama

def generate_insights(analysis_report, model="gemma3:1b"):
    """Generate evidence-based explanations using a local Gemma model."""

    prompt = f"""
    You are OpenLens AI, a careful data analyst.

    Analyze the following computed dataset statistics.
    Explain the findings in simple language.
    Do not invent values or claim causation.
    If evidence is insufficient, clearly say so.

    Dataset analysis:
    {analysis_report}

    Return:
    1. Three important observations
    2. Potential data quality issues
    3. Two actionable recommendations
    """

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]
'''

(PROJECT_DIR / "llm_client.py").write_text(
    llm_client_code,
    encoding="utf-8"
)

print("llm_client.py created successfully! ✅")

llm_client.py created successfully! ✅


In [15]:
from pathlib import Path

app_path = PROJECT_DIR / "app.py"
app_code = app_path.read_text(encoding="utf-8")

old_section = '''with tab4:
    st.subheader("Evidence-based AI Insights")

    st.write(
        "This report uses calculated dataset statistics. "
        "AI-generated explanations will be connected through Gemma."
    )

    st.markdown("**Dataset findings**")
    st.write(f"- Dataset contains {report['rows']} rows and {report['columns']} columns.")
    st.write(f"- {report['missing_cells']} missing cells were found.")
    st.write(f"- {report['duplicate_rows']} duplicate rows were found.")

    if report["numeric_summary"]:
        summary = pd.DataFrame(report["numeric_summary"]).T
        st.dataframe(summary, use_container_width=True)

    st.warning(
        "These are deterministic statistical findings, not live AI-generated responses yet."
    )
'''

new_section = '''with tab4:
    from llm_client import generate_insights

    st.subheader("🤖 Gemma AI Data Analyst")
    st.write("Generate explanations from the dataset's calculated statistics.")

    st.markdown("**Deterministic findings**")
    st.write(f"- Dataset contains {report['rows']} rows and {report['columns']} columns.")
    st.write(f"- {report['missing_cells']} missing cells were found.")
    st.write(f"- {report['duplicate_rows']} duplicate rows were found.")

    if st.button("✨ Generate AI Insights"):
        with st.spinner("Gemma is analyzing the evidence..."):
            try:
                insights = generate_insights(report)
                st.markdown("### AI-generated insights")
                st.write(insights)
                st.success("Generated using the configured local model.")
            except Exception as exc:
                st.error(
                    "Could not connect to Gemma. Make sure Ollama is installed, "
                    "running, and the configured model is downloaded."
                )
                st.caption(str(exc))
'''

if old_section not in app_code:
    print("Expected section nahi mila. app.py ko manually update karna hoga.")
else:
    app_path.write_text(
        app_code.replace(old_section, new_section),
        encoding="utf-8"
    )
    print("app.py updated successfully! ✅")

app.py updated successfully! ✅


In [23]:
requirements = """streamlit
pandas
numpy
plotly
ollama
"""

(PROJECT_DIR / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print("requirements.txt created successfully! ✅")

requirements.txt created successfully! ✅


In [26]:
readme = """# 🔍 OpenLens AI
**Your Data. Your Evidence.**

OpenLens AI is an open-source, AI-powered data analysis tool that transforms CSV files into clear insights. It detects data quality issues, visualizes patterns, and uses Google's Gemma model to explain findings.

## ✨ Features
- 📂 Upload and preview CSV datasets
- 🧹 Detect missing values and duplicate rows
- 📊 Explore statistics, outliers, and interactive charts
- 🤖 Generate AI-powered insights using Gemma
- 📥 Export analysis reports as JSON

## 🛠️ Tech Stack
Python • Pandas • NumPy • Streamlit • Plotly • Ollama • Gemma

## 🚀 Quick Start
1. Install dependencies: `pip install -r requirements.txt`
2. Install Ollama: https://ollama.com/download/windows
3. Download the model: `ollama pull gemma3:1b`
4. Launch: `python -m streamlit run app.py`

## 🧠 AI Contribution
Python computes dataset statistics; Gemma interprets those statistics to generate explanations and recommendations. AI-generated results should be verified against the data.

## 📂 Project Structure
- `app.py` — Streamlit dashboard
- `analyzer.py` — Data quality and statistical analysis
- `llm_client.py` — Gemma integration
- `data/` — Sample CSV dataset
- `requirements.txt` — Python dependencies

## 📜 License
MIT License. See `LICENSE` for details.
"""

(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
print("Short README.md created successfully!")

Short README.md created successfully!


In [29]:
from pathlib import Path

PROJECT_DIR = Path.home() / "OneDrive" / "Desktop" / "MLH"

for file in [
    "app.py",
    "analyzer.py",
    "llm_client.py",
    "README.md",
    "requirements.txt",
    "LICENSE",
    "data/sample_employees.csv"
]:
    path = PROJECT_DIR / file
    print(f"{'✅' if path.exists() else '❌'} {file}")

✅ app.py
✅ analyzer.py
✅ llm_client.py
✅ README.md
✅ requirements.txt
✅ LICENSE
✅ data/sample_employees.csv


In [28]:
license_text = """MIT License

Copyright (c) 2026 OpenLens AI Contributors

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

(PROJECT_DIR / "LICENSE").write_text(license_text, encoding="utf-8")

print("✅ LICENSE file created successfully!")

✅ LICENSE file created successfully!


In [30]:
print("Project folder:", PROJECT_DIR)
print("GitHub repository:", "https://github.com/rishi-prajapati-git/OpenLens-AI.git")

Project folder: C:\Users\praja\OneDrive\Desktop\MLH
GitHub repository: https://github.com/rishi-prajapati-git/OpenLens-AI.git
